# local group six model posteriors

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Observed Local Group: six trained NPE models

**Application only.** This notebook produces nominal and observational-input-mixture posteriors for each of the six registered final models. It does not train, refit scalers, change saved splits, rerun test-set evaluation, apply a stellar Delta correction, or combine the six posteriors into a consensus.

**Scientific status: labelled approximate observational scenario**, explicitly authorized on . Kinematics reuse the latest completed prior LG uncertainty analysis. Stellar masses use the cited observational proxies below. The missing joint covariance, stellar-mass definition/IMF systematics, and chosen combined-model Hubble convention are assumptions, not measured zero uncertainties. Results must retain this label in the thesis.

Default `RUN_INFERENCE=False` permits a local, read-only artifact audit without loading trained pickles. Scientific inference requires the matching pinned Roihu GPU environment. Run every cell in order; the supplied Slurm script enables inference explicitly. All generated files go into a new timestamped `out/local_group_final/` directory.


In [ ]:
RUN_INFERENCE = False
PROJECT_DIR = ""  # auto-discover, or absolute project directory
N_NOMINAL = 100_000
N_OBSERVATIONS = 5_000
N_CONDITIONAL = 20
CONDITION_BATCH_SIZE = 128
OBSERVATION_SEED = 26100301
POSTERIOR_SEED_BASE = 26100310
COMBINED_H0 = 68.1  # explicit new reference convention; not inferred from training
STABILITY_FRACTION_OF_WIDTH68 = 0.02  # diagnostic tolerance, not a significance test

## Target and interpretation

The mass target is `log_m200c`, with support **11.5–14.5 dex**: common host M200c for a same-host pair; sum of the two host M200c values for a different-host pair. MW and M31 are distinct main haloes, so the intended LG quantity is **M200c,MW + M200c,M31**.

However, each network was trained on a mixture of same-host and different-host pairs and has **no host-class conditioning input**. Its output is the learned mixed-population target distribution at LG-like features, interpreted as a summed-halo analogue with this limitation. It is not a verified different-host-only posterior or a single-halo mass measurement. The six results are dependent conditional inferences under different training/feature assumptions.

The BoxUniform restricts support; it does not replace the empirical catalogue target distribution with a uniform population prior.


In [ ]:
import os, sys, json, hashlib, random, shutil, warnings
from pathlib import Path
from datetime import datetime, timezone
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
from scipy.stats import truncnorm
from IPython.display import display, Markdown

def find_project(explicit):
    if explicit:
        candidates = [Path(explicit)]
    elif os.environ.get('LG_PROJECT'):
        candidates = [Path(os.environ['LG_PROJECT'])]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / 'out/final_six/model_registry_v1.json').is_file():
            return p.resolve()
    raise FileNotFoundError('Set PROJECT_DIR to the archived project or Roihu project root')

PROJECT = find_project(PROJECT_DIR)
sys.path.insert(0, str(PROJECT / 'src'))
sys.path.insert(0, str(PROJECT))
from lg_npe.final_design import sha256, load_model, require_saved_model_environment, load_final_frames, frame_digest, TARGET_DEFINITION
from lg_npe.features import feature_names, add_stellar_representations
from lg_npe.inference import posterior_draws
from lg_npe.registry import environment_record, write_json
from lg_npe.controlled_training import validate_final_config
from analogues.review_fixes import lg_observation, draw_noisy_feature, lg_to_peculiar, H0_BY_SIM

NOTEBOOK_DIR = PROJECT / 'notebooks/final'
EVIDENCE_PATH = NOTEBOOK_DIR / 'local_group_observation_evidence.json'
EVIDENCE = json.loads(EVIDENCE_PATH.read_text(encoding='utf-8'))
MODEL_KEYS = [f'{d}_{f}' for d in ('flamingo','tng','combined') for f in ('dynamics','stellar_sum')]
SUPPORT = (11.5, 14.5)
KIN_NAMES = ['separation', 'vr', 'vt']
assert RUN_INFERENCE in (True, False)
assert all(isinstance(x, int) and x > 0 for x in (N_NOMINAL,N_OBSERVATIONS,N_CONDITIONAL,CONDITION_BATCH_SIZE))
assert N_OBSERVATIONS >= 2000 and N_NOMINAL >= 100_000
assert N_OBSERVATIONS * N_CONDITIONAL >= 100_000
assert np.isfinite(COMBINED_H0) and COMBINED_H0 > 0
assert 0 < STABILITY_FRACTION_OF_WIDTH68 < 1
H0 = {**H0_BY_SIM, 'combined': float(COMBINED_H0)}
print('Project:', PROJECT, '| inference enabled:', RUN_INFERENCE)

## Observational inputs

| Quantity | Adopted value | Uncertainty parameter | Distribution used | Unit | Origin and reference |
|---|---:|---:|---|---|---|
| Galactocentric separation | 765 | 15 | Normal, truncated below 0 | kpc | **Recovered:** `lg_observation`, latest Salomon marginalized analysis; Li et al. (2021) distance plus project conversion/widening |
| Physical radial velocity | −109.3 | 4.4 | Normal | km/s | **Recovered:** van der Marel et al. (2012) and prior helper |
| Tangential speed | 82.4 | 31.2 | Normal, truncated below 0 | km/s | **Recovered approximation:** Salomon et al. (2021) and prior helper |
| MW stellar mass | 6.08 × 10¹⁰ | 1.14 × 10¹⁰ | Positive-truncated normal | Msun | **Literature application now:** Licquia & Newman (2015), already identified in prior decision record |
| M31 stellar mass, <30 kpc | 10.3 × 10¹⁰ | −1.7 / +2.3 × 10¹⁰ | Piecewise log-normal quantile approximation (defined below) | Msun | **Literature application now:** Sick et al. (2015), also discussed in prior decision log |

For truncated normals the listed values are the underlying **location/scale**, not truncated moments. The fixed nominal input uses the listed central values.

**Recovered conflicts.** `inference_uncertainty.ipynb` and the completed `npe_model_family_comparison.ipynb`/`pickles_v3/npe_model_family_lg_summary.csv` use Salomon2021 for marginalized results. The older van der Marel tangential value 17 km/s is included there only as a point scenario: its 34.3 km/s upper limit is not a Gaussian standard deviation. Stale 761 ± 11 comments refer to Li's **heliocentric Cepheid** distance. The executed helper instead uses a Galactocentric 765 kpc and a project-adopted 15 kpc spread, described in `DECISION_LOG.md`; 15 is not Li's quoted error. The old `log_mass_ratio` halo proxy is excluded.

**Velocity convention.** Existing `lg_to_peculiar` implements `vr_pec = vr_physical − H0 × separation/1000`, using the **same distance realization** before the stored scaler. FLAMINGO uses 68.1 and TNG 67.74 km/s/Mpc, as in the prior project. No combined-model observation convention was recorded. This application explicitly chooses **68.1** as a reference for both combined models, exposed as `COMBINED_H0`; it is not a fitted effective H0 or an average of posterior results. At 765 kpc the two single-simulation conversions differ by 0.2754 km/s. All stellar and physical kinematic realizations are shared across models.

**References (cite these in the thesis):**

* Li et al. (2021), *A sub-2% Distance to M31 from Photometrically Homogeneous Near-Infrared Cepheid Period-Luminosity Relations Measured with the Hubble Space Telescope*, ApJ **920**, 84. [DOI 10.3847/1538-4357/ac1597](https://doi.org/10.3847/1538-4357/ac1597), [arXiv:2107.08029](https://arxiv.org/abs/2107.08029).
* van der Marel et al. (2012), *The M31 Velocity Vector. II. Radial Orbit Towards the Milky Way and Implied Local Group Mass*, ApJ **753**, 8. [DOI 10.1088/0004-637X/753/1/8](https://doi.org/10.1088/0004-637X/753/1/8), [arXiv:1205.6864](https://arxiv.org/abs/1205.6864).
* Salomon et al. (2021), *The proper motion of Andromeda from Gaia eDR3: confirming a nearly radial orbit*, MNRAS **507**, 2592. [DOI 10.1093/mnras/stab2253](https://doi.org/10.1093/mnras/stab2253), [arXiv:2012.09204](https://arxiv.org/abs/2012.09204).
* Licquia, T. C. & Newman, J. A. (2015), *Improved Estimates of the Milky Way's Stellar Mass and Star Formation Rate from Hierarchical Bayesian Meta-Analysis*, ApJ **806**, 96. [DOI 10.1088/0004-637X/806/1/96](https://doi.org/10.1088/0004-637X/806/1/96), [arXiv:1407.1078](https://arxiv.org/abs/1407.1078).
* Sick et al. (2015), *The Stellar Mass of M31 as Inferred by the Andromeda Optical & Infrared Disk Survey*, IAU Symposium **311**, 82–85. [DOI 10.1017/S1743921315003440](https://doi.org/10.1017/S1743921315003440), [arXiv:1410.0017](https://arxiv.org/abs/1410.0017).


## Stellar-mass definitions

Licquia & Newman report disk plus bulge/bar mass under a Kroupa IMF and exponential-disk assumptions; their total posterior is well described by a Gaussian. The small stellar halo is omitted. This retains the estimate already identified in the prior project. McMillan (2017), MNRAS **465**, 76–94 ([DOI 10.1093/mnras/stw2759](https://doi.org/10.1093/mnras/stw2759), [arXiv:1608.00971](https://arxiv.org/abs/1608.00971)), reports (5.43 ± 0.57) × 10¹⁰ Msun for his bulge/disk dynamical model. Its stellar-to-halo abundance-matching prior makes it less attractive for an input intended to constrain halo mass independently.

Sick et al. fit stellar populations to M31 photometry and report bulge+disk mass **within 30 kpc**, with usable asymmetric errors. Tamm et al. (2012), *Stellar mass map and dark matter distribution in M31*, A&A **546**, A4 ([DOI 10.1051/0004-6361/201220065](https://doi.org/10.1051/0004-6361/201220065), [arXiv:1208.5712](https://arxiv.org/abs/1208.5712)), include nucleus, bulge, disk, young disk and stellar halo. Their (10–15) × 10¹⁰ range spans stellar-population models; it is not a measurement PDF. We do not turn it into a Gaussian or uniform distribution. Sick is chosen for its explicit photometric mass/error estimate, while retaining the aperture limitation.

Neither choice is exactly the simulated **all-bound stellar-mass** definition. Extended halo stars, component/aperture choices, stellar evolution/remnants, IMF and stellar-population assumptions remain unmatched. No numerical halo or IMF correction is invented; the full M31 IMF conversion cannot be established from the short source. These mass proxies are only an approximate observational analogue, and their quoted uncertainties do not capture the full definition mismatch.

**New M31 sampling assumption:** draw $z\sim N(0,1)$, then

$$M_{31}=M_0\exp\left[ z\begin{cases}\ln(M_0/(M_0-\sigma_-)), &z<0\\\ln((M_0+\sigma_+)/M_0),&z\geq0.\end{cases}\right]$$

This is positive, has median $M_0$, and maps $z=-1,+1$ to the reported lower/upper bars. Treating those bars as one-normal-sigma quantiles is **our approximation**: their confidence interpretation and full sampling distribution were not established from the short proceedings source. It preserves the reported asymmetry without claiming to recover the authors' posterior.

In physical coordinates we retain prior independent kinematic draws and add independent MW/M31 mass draws. No empirical covariance matrix was recovered. Shared IMF/SPS systematics, M31 distance–stellar-mass–proper-motion dependence and solar-frame covariance are **omitted, not measured absent**. The distance–peculiar-radial-velocity covariance induced by the established conversion is included exactly (up to its existing float32 cast).

The equal-weight pool approximates $\int q_S(\theta\mid x)g_{\mathrm{approx}}(x\mid y)\,dx$. This is the user-requested **observational-input mixture**, conditional on this approximation. It need not equal an exact simulation-population Bayesian posterior: that would generally require a measurement likelihood and the population feature density $p_S(x)$. We make no unvalidated importance weighting claim.


In [ ]:
REFERENCES = {
 'distance': {'citation': 'Li et al. 2021, ApJ 920, 84', 'doi': '10.3847/1538-4357/ac1597', 'arxiv': '2107.08029', 'project_adaptation': '761+/-11 heliocentric -> nominal 765 Galactocentric; project spread 15'},
 'vr': {'citation': 'van der Marel et al. 2012, ApJ 753, 8', 'doi': '10.1088/0004-637X/753/1/8', 'arxiv': '1205.6864'},
 'vt': {'citation': 'Salomon et al. 2021, MNRAS 507, 2592', 'doi': '10.1093/mnras/stab2253', 'arxiv': '2012.09204'},
 'mw_stellar': {'citation': 'Licquia & Newman 2015, ApJ 806, 96', 'doi': '10.1088/0004-637X/806/1/96', 'arxiv': '1407.1078', 'components': 'disk + bulge/bar; Kroupa IMF; stellar halo omitted'},
 'm31_stellar': {'citation': 'Sick et al. 2015, IAU Symposium 311, 82-85', 'doi': '10.1017/S1743921315003440', 'arxiv': '1410.0017', 'components': 'photometric bulge+disk within 30 kpc; not all-bound halo-inclusive'},
 'alternatives_inspected': [{'citation':'McMillan 2017, MNRAS 465, 76-94','doi':'10.1093/mnras/stw2759'}, {'citation':'Tamm et al. 2012, A&A 546, A4','doi':'10.1051/0004-6361/201220065'}]
}
OBS, UNC = lg_observation('Salomon2021')
assert [OBS[k] for k in KIN_NAMES] == [765., -109.3, 82.4]
assert [UNC[k] for k in KIN_NAMES] == [15., 4.4, 31.2]
MASS = dict(mw=6.08e10, mw_sigma=1.14e10, m31=10.3e10, m31_lower=1.7e10, m31_upper=2.3e10)
OBSERVATION_SPEC = dict(
 scenario='Salomon2021_plus_literature_stellar_proxies_approx_v1',
 status='user-authorized labelled approximation, not validated joint likelihood',
 physical_nominal={k:OBS[k] for k in KIN_NAMES}, kinematic_scales={k:UNC[k] for k in KIN_NAMES},
 kinematic_distribution='independent normal vr; lower-zero truncated normal separation/vt, locations/scales not truncated moments',
 stellar=MASS, mw_distribution='normal(mean=mw,sd=mw_sigma) conditioned on positive mass',
 m31_distribution='z~N(0,1); M=m31*exp(z*log(m31/(m31-m31_lower))) if z<0, else m31*exp(z*log((m31+m31_upper)/m31))',
 m31_error_interpretation='reported bars treated as z=-1,+1 quantiles only as an explicit approximation; source confidence level not verified',
 covariance='physical coordinates and stellar masses independent approximation; full observed covariance unavailable; same-draw distance used for vr peculiar conversion',
 omitted_systematics=['shared IMF/SPS','distance-mass-proper-motion covariance','solar-frame covariance','all-bound versus aperture/component definitions'],
 H0_km_s_Mpc=H0, combined_H0_status='new explicit reference convention, no prior combined LG convention recovered',
 references=REFERENCES, recovery=EVIDENCE['recovery'], user_decision=EVIDENCE['user_decision'])

def observation_frame(kinematics, mw, m31):
    a = np.asarray(kinematics, dtype=float)
    assert a.ndim == 2 and a.shape[1] == 3
    f = pd.DataFrame(a, columns=KIN_NAMES)
    f['mstar_a'], f['mstar_b'] = mw, m31
    # Stable dummy IDs only satisfy the existing ordering helper; never model inputs.
    f['id_a'], f['id_b'] = 0, 1
    if not np.isfinite(f.to_numpy()).all() or (f[['separation','vt','mstar_a','mstar_b']] <= 0).any().any():
        raise ValueError('Unphysical/non-finite observational realization')
    return add_stellar_representations(f)

def asymmetric_positive_mass(z, centre, lower, upper):
    assert centre > lower > 0 and upper > 0
    z = np.asarray(z, dtype=float)
    scale = np.where(z < 0, np.log(centre/(centre-lower)), np.log((centre+upper)/centre))
    return centre * np.exp(z * scale)

def observation_realizations(n, seed):
    rng = np.random.default_rng(seed)
    kin = np.column_stack([draw_noisy_feature(k,OBS[k],UNC[k],n,rng) for k in KIN_NAMES])
    mw = truncnorm.rvs(-MASS['mw']/MASS['mw_sigma'],np.inf,loc=MASS['mw'],scale=MASS['mw_sigma'],size=n,random_state=rng)
    m31 = asymmetric_positive_mass(rng.normal(size=n), MASS['m31'], MASS['m31_lower'], MASS['m31_upper'])
    return observation_frame(kin,mw,m31)

def model_raw(frame, names, population):
    return lg_to_peculiar(frame[names].to_numpy(float), names, H0[population])

nominal_frame = observation_frame([[OBS[k] for k in KIN_NAMES]], MASS['mw'], MASS['m31'])
nominal_rows = []
for population in ('flamingo','tng','combined'):
    raw = model_raw(nominal_frame, ['separation','vr','vt','log_stellar_sum'],population)[0]
    nominal_rows.append(dict(population=population,H0=H0[population],separation_kpc=raw[0],vr_peculiar_km_s=raw[1],vt_km_s=raw[2],log_stellar_sum=raw[3]))
display(pd.DataFrame(nominal_rows))
print('Unrounded nominal log10 stellar sum:', float(nominal_frame.log_stellar_sum.iloc[0]))

## Check models and inputs

Require exactly the registered six models, archived run-manifest/model hashes, reference data/split identity, unchanged design/source hashes, configuration, feature order and target. The evidence JSON pins the registry reviewed locally. Actual scaler state and posterior support are checked after environment validation on Roihu; locally their availability is established by the saved model construction, not by unsafe deserialization.


In [ ]:
REGISTRY_PATH = PROJECT / 'out/final_six/model_registry_v1.json'
REGISTRY = json.loads(REGISTRY_PATH.read_text(encoding='utf-8'))
assert set(REGISTRY) == set(MODEL_KEYS), 'Registry must contain exactly the final six models'
assert REGISTRY == EVIDENCE['frozen_registry']
assert sha256(REGISTRY_PATH) == EVIDENCE['registry_sha256'], 'Changed registry'
assert sha256(PROJECT/'analogues/review_fixes.py') == EVIDENCE['review_fixes_sha256'], 'Changed observation helper'
BUNDLE = PROJECT / 'data/derived/final_experiment_v1'
IDENTITY = json.loads((BUNDLE/'identity.json').read_text())
SPLIT_HASH = sha256(BUNDLE/'splits.npz')
MANIFESTS, AUDIT, INPUT_HASHES = {}, [], {}
for key in MODEL_KEYS:
    population, family = key.split('_',1)
    expected_mass = 'none' if family == 'dynamics' else 'stellar_sum'
    domains = ['flamingo','tng'] if population == 'combined' else [population]
    run = (PROJECT / REGISTRY[key]).resolve()
    assert run.is_relative_to(PROJECT), 'Registry path escapes project'
    model_path, manifest_path = run/'model.pkl', run/'run_manifest.json'
    assert model_path.is_file() and manifest_path.is_file(), key
    manifest = json.loads(manifest_path.read_text())
    assert sha256(manifest_path) == EVIDENCE['run_manifest_sha256'][key], 'Changed run manifest'
    actual = sha256(model_path)
    assert actual == manifest['model_sha256'] == EVIDENCE['model_sha256'][key], key
    c = manifest['config']
    validate_final_config(c)
    config_path = PROJECT / 'configs/final' / (key+'.json')
    assert json.loads(config_path.read_text()) == c, 'Archived configuration differs: '+key
    INPUT_HASHES[str(config_path.relative_to(PROJECT))] = sha256(config_path)
    assert c['experiment_id'] == 'final_v1_' + key
    assert c['mass_feature'] == expected_mass and c['training_domains'] == domains
    assert c['target'] == 'm200c' and c['observation_noise'] == 'none'
    assert (c['npe']['prior_low'],c['npe']['prior_high']) == SUPPORT
    assert manifest['data_identity'] == IDENTITY and manifest['split_sha256'] == SPLIT_HASH
    assert sha256(design_path(c['design_record'])) == manifest['design_sha256']
    INPUT_HASHES[str(design_path(c['design_record']).relative_to(PROJECT))] = manifest['design_sha256']
    for relative, digest in manifest['source_sha256'].items():
        assert sha256(PROJECT/relative) == digest, f'Changed frozen source: {relative}'
        INPUT_HASHES[relative] = digest
    names = feature_names('dynamics',expected_mass)
    assert names == KIN_NAMES + ([] if family == 'dynamics' else ['log_stellar_sum'])
    MANIFESTS[key] = manifest
    AUDIT.append(dict(model=key,model_path=str(model_path),sha256=actual,features=names,target='log_m200c',support=SUPPORT,scaler='saved train-only StandardScaler; inspect on Roihu'))
for p in [REGISTRY_PATH,EVIDENCE_PATH,BUNDLE/'identity.json',BUNDLE/'splits.npz',PROJECT/'analogues/review_fixes.py',NOTEBOOK_DIR/'local_group_six_model_posteriors.ipynb']:
    INPUT_HASHES[str(p.relative_to(PROJECT))] = sha256(p)
display(pd.DataFrame(AUDIT))
print('All six model and manifest hashes verified; source, split, config, feature and target audit passed.')

## Monte Carlo design and runtime gate

Defaults: **100,000 nominal draws per model** and **5,000 shared observational realizations × 20 conditional draws = 100,000 mixture draws per model**. Equal conditional counts implement equal observational weights. Shared observations reduce extraneous input-Monte-Carlo differences between model comparisons; they do not make the model posteriors independent. No observation is removed for exceeding a training range.

Seeds: observations `26100301`; for model index $i$ in the displayed order, nominal `26100310 + 2i`, mixture `26100311 + 2i`. Condition batches of 128 reuse the project's `posterior_draws` implementation. Deterministic seeds are recorded; bitwise GPU reproducibility across hardware/library changes is not promised.

Progressive observational prefixes and disjoint halves compare five quantiles to the full mixture, normalized by its 68% width. The declared 2%-of-width tolerance is a numerical diagnostic, not a scientific significance or materiality threshold. If the largest-prefix/half diagnostic fails, the output is explicitly flagged **MC stability not established**. Do not claim negligible Monte Carlo noise merely because the job completed.


In [ ]:
MODELS, SCALER_RECORDS = {}, {}
if RUN_INFERENCE:
    require_saved_model_environment()  # exact Python/package pins and CUDA, before any pickle
    import torch
    for key in MODEL_KEYS:
        m = load_model(PROJECT/REGISTRY[key]/'model.pkl')
        c = MANIFESTS[key]['config']
        names = feature_names('dynamics',c['mass_feature'])
        assert m['config'] == c and m['feature_names'] == names and m['target'] == 'log_m200c', key
        scaler = m['scaler']
        assert type(scaler).__name__ == 'StandardScaler'
        assert scaler.with_mean and scaler.with_std and scaler.n_features_in_ == len(names)
        for a in (scaler.mean_,scaler.scale_,scaler.var_):
            assert np.asarray(a).shape == (len(names),) and np.isfinite(a).all()
        assert np.all(scaler.scale_ > 0) and np.all(scaler.var_ >= 0)
        prior = m['posterior'].prior
        base = getattr(prior,'base_dist',prior)
        low = np.asarray(base.low.detach().cpu()).ravel()
        high = np.asarray(base.high.detach().cpu()).ravel()
        np.testing.assert_array_equal(low,[SUPPORT[0]])
        np.testing.assert_array_equal(high,[SUPPORT[1]])
        assert all(p.device.type == 'cuda' for p in m['density_estimator'].parameters())
        m['density_estimator'].eval()
        MODELS[key] = m
        SCALER_RECORDS[key] = dict(class_name=type(scaler).__name__,features=names,
            mean=scaler.mean_,scale=scaler.scale_,variance=scaler.var_,n_samples_seen=scaler.n_samples_seen_,
            provenance='stored train-only equal-domain-weighted scaler; never refit; internal training z_score_x=none',
            model_sha256=MANIFESTS[key]['model_sha256'],posterior_support=[low.tolist(),high.tolist()])
    print('All saved configurations, feature orders, scalers and live posterior supports verified.')
else:
    print('AUDIT ONLY: no saved model loaded, observational MC generated, GPU used or output directory created.')

## Training-range diagnostics

Only the **training indices** are used for these diagnostics. The original catalogue loaders and fixed 12-column identity protect the population; no new split or model evaluation is performed. Combined min/max is the union of its two training populations; reported quantiles use equal domain weights, matching the training objective. Scaled values use each stored scaler.

The compact Windows archive omits catalogue inputs needed by `load_final_frames`. If those inputs remain unavailable on the execution host, diagnostics explicitly report `unavailable` while inference may continue. A changed identity or other inconsistency is an error, not a missing-data excuse. Being inside all marginal ranges does not establish joint support, and being inside the BoxUniform target support says nothing about input extrapolation.


In [ ]:
def frozen_training_frames():
    try:
        frames = load_final_frames(PROJECT)
    except FileNotFoundError as exc:
        return None, 'unavailable: original catalogue dependency missing: ' + str(exc)
    for domain, frame in frames.items():
        assert frame_digest(frame) == IDENTITY[domain], f'Changed {domain} catalogue identity'
    with np.load(BUNDLE/'splits.npz',allow_pickle=False) as split:
        training = {}
        for d, frame in frames.items():
            ix = split[d+'_train']
            assert np.issubdtype(ix.dtype,np.integer) and len(np.unique(ix)) == len(ix)
            assert ix.min() >= 0 and ix.max() < len(frame)
            training[d] = frame.iloc[ix]
    return training, 'available: verified frozen training rows; no validation or test rows used'

def training_quantiles(values, domain_sizes):
    # Inverse weighted empirical CDF; every domain receives equal total weight.
    weights = np.concatenate([np.full(n,1/(len(domain_sizes)*n)) for n in domain_sizes])
    order = np.argsort(values,kind='stable')
    cdf = np.cumsum(weights[order]); cdf[-1] = 1.
    return values[order][np.searchsorted(cdf,[.01,.05,.5,.95,.99])]

def support_diagnostics(models, training, status, nominal, observations):
    rows = []
    for key in MODEL_KEYS:
        m = models[key]; names = m['feature_names']; population = key.split('_',1)[0]
        raw0, raw = model_raw(nominal,names,population), model_raw(observations,names,population)
        scaled0 = m['scaler'].transform(raw0)[0]
        domains = m['config']['training_domains']
        train = None if training is None else np.concatenate([training[d][names].to_numpy(float) for d in domains])
        joint_outside = None if train is None else ((raw < train.min(0)) | (raw > train.max(0))).any(axis=1).mean()
        for j,name in enumerate(names):
            row = dict(model=key,feature=name,nominal_raw=float(raw0[0,j]),nominal_scaled=float(scaled0[j]),
                H0=H0[population],status=status,any_feature_outside_fraction=joint_outside,
                n_observations=len(raw),quantile_definition='equal-domain-weighted inverse empirical CDF')
            if train is not None:
                a=train[:,j]; lo,hi=float(a.min()),float(a.max())
                q=training_quantiles(a,[len(training[d]) for d in domains])
                row.update(train_count=len(a),train_min=lo,train_max=hi,
                    train_q01=q[0],train_q05=q[1],train_q50=q[2],train_q95=q[3],train_q99=q[4],
                    nominal_outside=bool(raw0[0,j]<lo or raw0[0,j]>hi),
                    below_fraction=float((raw[:,j]<lo).mean()),above_fraction=float((raw[:,j]>hi).mean()),
                    outside_fraction=float(((raw[:,j]<lo)|(raw[:,j]>hi)).mean()))
            rows.append(row)
    return pd.DataFrame(rows)

if RUN_INFERENCE:
    training, SUPPORT_STATUS = frozen_training_frames()
    OBSERVATIONS = observation_realizations(N_OBSERVATIONS,OBSERVATION_SEED)
    support_table = support_diagnostics(MODELS,training,SUPPORT_STATUS,nominal_frame,OBSERVATIONS)
    display(support_table)
    if training is None: warnings.warn(SUPPORT_STATUS)
    del training

## Generate and preserve both posteriors for every model

Each uncertainty NPZ retains a `(conditional_draw, observation)` matrix as well as flattened log and physical-mass draws. Its columns correspond to `observational_inputs.npz`; weights are equal. Raw and scaled features, seeds, model hashes and scenario labels accompany the samples. Flattening uses C order and is recorded explicitly.

The output directory is new and exclusive. `manifest.json` starts as `running` and becomes `complete` only after all tables, figures and markdown are written. A failed execution leaves a partial run labelled accordingly; do not treat it as a completed scientific result.


In [ ]:
def posterior_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def validate_draws(a, expected_shape):
    a = np.asarray(a)
    assert a.shape == expected_shape, (a.shape,expected_shape)
    assert np.isfinite(a).all() and np.all((a >= SUPPORT[0]) & (a <= SUPPORT[1]))
    return a

def summarize_draws(a, key, kind, seed, model_path, n_observations):
    a = np.asarray(a).reshape(-1).astype(np.float64)
    q = np.quantile(a,[.025,.16,.5,.84,.975])
    mass = np.power(10.,a)  # transform EVERY draw, then summarize in physical space
    mq = np.quantile(mass,[.025,.16,.5,.84,.975])
    population,family = key.split('_',1)
    return dict(model=key,training_population=population,feature_set=family,inference_type=kind,
        posterior_draws=len(a),observational_mc_count=n_observations,seed=seed,
        observation_seed=OBSERVATION_SEED if n_observations > 1 else None,model_path=str(model_path),
        scenario=OBSERVATION_SPEC['scenario'],target='mixed-host log_m200c; intended LG analogue sum M200c_MW+M200c_M31',
        log_mean=a.mean(),log_std=a.std(ddof=0),log_q025=q[0],log_q16=q[1],log_median=q[2],log_q84=q[3],log_q975=q[4],
        log_width68=q[3]-q[1],log_width95=q[4]-q[0],
        mass_q025_msun=mq[0],mass_q16_msun=mq[1],mass_median_msun=mq[2],mass_q84_msun=mq[3],mass_q975_msun=mq[4])

def stability_rows(matrix, key):
    n=matrix.shape[1]
    probs=np.array([.025,.16,.5,.84,.975]); full=np.quantile(matrix,probs)
    width=full[3]-full[1]
    assert width > 0
    prefixes=sorted(set([500,1000,2000,n//2,n]))
    selections=[(f'prefix_{k}',slice(0,k),k) for k in prefixes if k <= n]
    selections += [('first_half',slice(0,n//2),n//2),('second_half',slice(n//2,n),n-n//2)]
    rows=[]
    for label,selection,count in selections:
        q=np.quantile(matrix[:,selection],probs)
        for p,value,reference in zip(probs,q,full):
            frac=abs(value-reference)/width
            rows.append(dict(model=key,subset=label,observational_mc_count=count,posterior_draws=count*matrix.shape[0],
                probability=p,quantile=value,full_quantile=reference,difference_dex=value-reference,
                abs_difference_over_full_width68=frac,tolerance=STABILITY_FRACTION_OF_WIDTH68,
                within_tolerance=bool(frac<=STABILITY_FRACTION_OF_WIDTH68)))
    return rows

DRAWS, summary_rows, stability_records = {}, [], []
if RUN_INFERENCE:
    timestamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    OUTPUT=PROJECT/'out/local_group_final'/timestamp
    OUTPUT.mkdir(parents=True,exist_ok=False)
    if os.environ.get('LG_JOB_DIR'):
        job_dir=Path(os.environ['LG_JOB_DIR']).resolve()
        assert job_dir.is_relative_to((PROJECT/'out/local_group_jobs').resolve()) and job_dir.is_dir()
        with (job_dir/'scientific_output.json').open('x',encoding='utf-8') as stream:
            json.dump({'output_directory':str(OUTPUT)},stream,indent=2)
    for source in [NOTEBOOK_DIR/'local_group_six_model_posteriors.ipynb',EVIDENCE_PATH]:
        shutil.copyfile(source,OUTPUT/source.name)
    MANIFEST=dict(schema='local-group-final-v1',status='running',timestamp_utc=timestamp,
        observations=OBSERVATION_SPEC,source_recovery=EVIDENCE,model_audit=AUDIT,
        run_manifests=MANIFESTS,scalers=SCALER_RECORDS,input_sha256=INPUT_HASHES,
        feature_units={'separation':'kpc','vr':'peculiar km/s','vt':'km/s','log_stellar_sum':'log10(Mstar_MW+Mstar_M31 in Msun)'},
        target_definition=TARGET_DEFINITION,target_interpretation='intended summed MW+M31 M200c analogue; models trained on mixed same/different-host targets without host class conditioning',
        posterior_support=SUPPORT,nominal_draws=N_NOMINAL,observational_mc_count=N_OBSERVATIONS,
        draws_per_observation=N_CONDITIONAL,marginalized_draws=N_OBSERVATIONS*N_CONDITIONAL,
        condition_batch_size=CONDITION_BATCH_SIZE,observation_seed=OBSERVATION_SEED,
        posterior_seeds={k:{'nominal':POSTERIOR_SEED_BASE+2*i,'observational_uncertainty':POSTERIOR_SEED_BASE+2*i+1} for i,k in enumerate(MODEL_KEYS)},
        environment=environment_record(),gpu_used=True,support_status=SUPPORT_STATUS,
        source_notebook=str(NOTEBOOK_DIR/'local_group_six_model_posteriors.ipynb'),
        executed_notebook=os.environ.get('LG_EXECUTED_NOTEBOOK'),
        reproducibility='fixed seeds in pinned environment; cross-hardware bitwise reproducibility not guaranteed',
        mixture='equal weights for shared independent observational realizations and equal conditional draw counts; no population reweighting',
        flatten_order='C; conditional_draw axis first, observation axis second',
        stability_tolerance_fraction_width68=STABILITY_FRACTION_OF_WIDTH68)
    write_json(OUTPUT/'manifest.json',MANIFEST)
    support_table.to_csv(OUTPUT/'local_group_support_diagnostics.csv',index=False)
    OBSERVATIONS.to_csv(OUTPUT/'observational_inputs.csv',index=False)
    np.savez_compressed(OUTPUT/'observational_inputs.npz',
        columns=np.array(OBSERVATIONS.columns,dtype=str),values=OBSERVATIONS.to_numpy(float),
        nominal=nominal_frame.to_numpy(float),observation_index=np.arange(N_OBSERVATIONS),
        weights=np.full(N_OBSERVATIONS,1/N_OBSERVATIONS),seed=OBSERVATION_SEED)
    try:
        for i,key in enumerate(MODEL_KEYS):
            m=MODELS[key]; names=m['feature_names']; population=key.split('_',1)[0]
            raw0=model_raw(nominal_frame,names,population)
            raw=model_raw(OBSERVATIONS,names,population)
            x0=m['scaler'].transform(raw0); x=m['scaler'].transform(raw)
            assert np.isfinite(x0).all() and np.isfinite(x).all()
            sn=POSTERIOR_SEED_BASE+2*i; su=sn+1
            posterior_seed(sn)
            nominal=validate_draws(posterior_draws(m['posterior'],x0,N_NOMINAL,1,'cuda'),(N_NOMINAL,1))[:,0]
            posterior_seed(su)
            matrix=validate_draws(posterior_draws(m['posterior'],x,N_CONDITIONAL,CONDITION_BATCH_SIZE,'cuda'),(N_CONDITIONAL,N_OBSERVATIONS))
            mixture=matrix.ravel(order='C')
            common=dict(model=key,model_sha256=MANIFESTS[key]['model_sha256'],feature_names=np.array(names),scenario=OBSERVATION_SPEC['scenario'])
            np.savez_compressed(OUTPUT/f'{key}_lg_nominal_draws.npz',theta_log10_msun=nominal,mass_msun=10.**nominal.astype(float),
                features_raw=raw0,features_scaled=x0,seed=sn,**common)
            np.savez_compressed(OUTPUT/f'{key}_lg_uncertainty_draws.npz',theta_log10_msun=mixture,mass_msun=10.**mixture.astype(float),
                theta_by_observation=matrix,observation_index=np.arange(N_OBSERVATIONS),features_raw=raw,features_scaled=x,
                observation_weights=np.full(N_OBSERVATIONS,1/N_OBSERVATIONS),seed=su,observation_seed=OBSERVATION_SEED,flatten_order='C',**common)
            DRAWS[key]={'nominal':nominal,'observational_uncertainty':mixture}
            for kind,values,seed,count in [('nominal',nominal,sn,1),('observational_uncertainty',mixture,su,N_OBSERVATIONS)]:
                summary_rows.append(summarize_draws(values,key,kind,seed,PROJECT/REGISTRY[key]/'model.pkl',count))
            stability_records.extend(stability_rows(matrix,key))
            print(key,': saved nominal and observational-input mixture')
        summary=pd.DataFrame(summary_rows)
        assert len(summary)==12 and not summary.duplicated(['model','inference_type']).any()
        summary.to_csv(OUTPUT/'local_group_posterior_summary.csv',index=False)
        stability=pd.DataFrame(stability_records)
        stability.to_csv(OUTPUT/'local_group_mc_stability.csv',index=False)
        checks=stability[stability['subset'].isin(['first_half','second_half',f'prefix_{max(N_OBSERVATIONS//2,2000)}'])]
        stability_pass=checks.groupby('model').within_tolerance.all().to_dict()
        MANIFEST['mc_stability_pass']=stability_pass
        if not all(stability_pass.values()):
            warnings.warn('MC stability not established for one or more models; retain flags and inspect before thesis claims.')
        display(summary)
        display(checks.groupby('model').abs_difference_over_full_width68.max().rename('largest_checked_quantile_difference_over_width68'))
    except Exception as exc:
        MANIFEST.update(status='failed',failure_type=type(exc).__name__,failure=str(exc))
        write_json(OUTPUT/'manifest.json',MANIFEST)
        raise

## Descriptive contrasts

Widths and median shifts below are measured in **dex**. Percentage width change uses the first/nominal width as denominator. These are descriptive, correlated model comparisons, not independent measurements or significance tests. A narrower interval need not be more accurate. Absolute shifts of medians and changes in interval width answer different questions; report both.


In [ ]:
def build_contrasts(summary):
    lookup=summary.set_index(['model','inference_type'])
    effects=[]; comparisons=[]
    for key in MODEL_KEYS:
        a=lookup.loc[(key,'nominal')]; b=lookup.loc[(key,'observational_uncertainty')]
        effects.append(dict(model=key,median_shift_dex=b.log_median-a.log_median,
            width68_change_dex=b.log_width68-a.log_width68,
            width68_percentage_change=100*(b.log_width68/a.log_width68-1),
            width95_change_dex=b.log_width95-a.log_width95,
            width95_percentage_change=100*(b.log_width95/a.log_width95-1)))
    pairs=[(f'{d}_dynamics',f'{d}_stellar_sum','add_stellar') for d in ('flamingo','tng','combined')]
    for family in ('dynamics','stellar_sum'):
        pairs += [(f'flamingo_{family}',f'tng_{family}','TNG_minus_FLAMINGO'),
                  (f'flamingo_{family}',f'combined_{family}','combined_minus_FLAMINGO'),
                  (f'tng_{family}',f'combined_{family}','combined_minus_TNG')]
    for kind in ('nominal','observational_uncertainty'):
        for ka,kb,label in pairs:
            a=lookup.loc[(ka,kind)]; b=lookup.loc[(kb,kind)]
            comparisons.append(dict(inference_type=kind,contrast=label,reference_model=ka,comparison_model=kb,
                median_difference_dex=b.log_median-a.log_median,width68_difference_dex=b.log_width68-a.log_width68,
                width68_percentage_change=100*(b.log_width68/a.log_width68-1),width95_difference_dex=b.log_width95-a.log_width95))
    return pd.DataFrame(effects),pd.DataFrame(comparisons)

if RUN_INFERENCE:
    effects,contrasts=build_contrasts(summary)
    effects.to_csv(OUTPUT/'local_group_observational_uncertainty_effects.csv',index=False)
    contrasts.to_csv(OUTPUT/'local_group_model_contrasts.csv',index=False)
    display(effects); display(contrasts)

## Six primary figures and two training-population comparisons

All eight figures use the same support-wide x limits and global y limits. Histograms use identical 0.02-dex bins and are normalized separately; no smoothing assumption or truth line is introduced. Each primary caption gives both medians and 68% intervals. Figure captions and labels retain the approximate-scenario and mixed-host interpretation. SVG and 300 dpi PNG files are saved. The same draw files and tables suffice to remake these figures without loading a model.


In [ ]:
def make_figures(draws, summary, output):
    import matplotlib.pyplot as plt
    plt.rcParams.update({'font.size':10,'axes.labelsize':11,'axes.titlesize':11,
        'svg.fonttype':'none','savefig.dpi':300,'axes.spines.top':False,'axes.spines.right':False})
    bins=np.linspace(SUPPORT[0],SUPPORT[1],151)
    hist={(key,kind):np.histogram(a,bins=bins,density=True)[0] for key,parts in draws.items() for kind,a in parts.items()}
    ymax=max(h.max() for h in hist.values())*1.10
    lookup=summary.set_index(['model','inference_type'])
    axis_label=r'$\log_{10}(M_{\rm LG,analogue}/M_\odot)$'
    population_titles={'flamingo':'FLAMINGO-trained','tng':'TNG-trained','combined':'Balanced combined'}
    figure_names=[]
    def finish(fig,name):
        for ext in ('svg','png'): fig.savefig(output/f'{name}.{ext}',bbox_inches='tight')
        figure_names.append(name)
        display(fig); plt.close(fig)
    for key in MODEL_KEYS:
        population,family=key.split('_',1)
        fig,ax=plt.subplots(figsize=(7.2,4.5))
        for kind,label,color,style in [('nominal','Nominal','#0072B2','--'),('observational_uncertainty','Input mixture (approx.)','#D55E00','-')]:
            ax.stairs(hist[(key,kind)],bins,label=label,color=color,linestyle=style,lw=1.7)
        ax.set(xlim=SUPPORT,ylim=(0,ymax),xlabel=axis_label,ylabel='Posterior density [dex$^{-1}$]',title=population_titles[population]+' | '+family.replace('_',' '))
        ax.legend(frameon=False)
        n=lookup.loc[(key,'nominal')]; u=lookup.loc[(key,'observational_uncertainty')]
        caption=(f'Nominal: {n.log_median:.3f} [{n.log_q16:.3f}, {n.log_q84:.3f}]; '
                 f'approx. mixture: {u.log_median:.3f} [{u.log_q16:.3f}, {u.log_q84:.3f}] dex (68%).\n'
                 'Summed-halo LG analogue; mixed-host-trained target. Stellar definition/covariance limitations apply.')
        fig.text(.08,.015,caption,fontsize=8,va='bottom')
        fig.tight_layout(rect=(0,.10,1,1))
        finish(fig,key+'_lg_posteriors')
    for kind in ('nominal','observational_uncertainty'):
        fig,axes=plt.subplots(1,3,figsize=(12.8,4.0),sharex=True,sharey=True)
        for ax,population in zip(axes,('flamingo','tng','combined')):
            for family,label,color,style in [('dynamics','Dynamics','#0072B2','--'),('stellar_sum','Stellar sum','#D55E00','-')]:
                ax.stairs(hist[(population+'_'+family,kind)],bins,label=label,color=color,linestyle=style,lw=1.6)
            ax.set(xlim=SUPPORT,ylim=(0,ymax),xlabel=axis_label,title=population_titles[population])
        axes[0].set_ylabel('Posterior density [dex$^{-1}$]'); axes[0].legend(frameon=False)
        label='Nominal inputs' if kind=='nominal' else 'Observational-input mixture (labelled approximation)'
        fig.suptitle(label)
        fig.text(.08,.015,'Summed-halo LG analogue; mixed-host-trained models. Six dependent conditional inferences; no consensus mass.',fontsize=8)
        fig.tight_layout(rect=(0,.07,1,.96))
        finish(fig,'local_group_comparison_'+kind)
    return figure_names

if RUN_INFERENCE:
    FIGURE_NAMES=make_figures(DRAWS,summary,OUTPUT)

## Markdown-ready findings and completed provenance

The following text is generated only from the actual posterior summaries. “Broadens/narrows” describes an observed numerical direction; percentage changes let the reader assess materiality. We do not define a post hoc “statistically significant” cutoff. The training-population comparison concerns shifts between conditional inferences, not a calibrated total simulation uncertainty.


In [ ]:
def result_markdown(summary,effects,contrasts,stability_pass,support_status):
    lookup=summary.set_index(['model','inference_type'])
    lines=['# Local Group application of six frozen models',
        '', '**Scenario: labelled approximate observational-input mixture.**',
        'Mass refers to the intended summed MW+M31 M200c analogue; each model was trained on mixed same/different-host targets.',
        'Quoted 68% intervals are equal-tailed. Results are dependent conditional inferences, not independent mass measurements.', '']
    for kind in ('nominal','observational_uncertainty'):
        lines += ['## '+kind.replace('_',' ').title(),'']
        for key in MODEL_KEYS:
            r=lookup.loc[(key,kind)]
            lines.append(f'- {key}: {r.log_median:.3f} [{r.log_q16:.3f}, {r.log_q84:.3f}] dex; '
                f'{r.mass_median_msun/1e12:.3f} [{r.mass_q16_msun/1e12:.3f}, {r.mass_q84_msun/1e12:.3f}] × 10^12 Msun.')
        lines.append('')
        for population in ('flamingo','tng','combined'):
            a=lookup.loc[(population+'_dynamics',kind)]; b=lookup.loc[(population+'_stellar_sum',kind)]
            change=100*(b.log_width68/a.log_width68-1)
            lines.append(f'- {population}: adding stellar information changes the 68% width by {change:+.1f}% '
                f'and the median by {b.log_median-a.log_median:+.3f} dex; this does not establish improved accuracy.')
        gaps={f:abs(lookup.loc[('flamingo_'+f,kind)].log_median-lookup.loc[('tng_'+f,kind)].log_median) for f in ('dynamics','stellar_sum')}
        lines.append(f'- Absolute FLAMINGO–TNG median disagreement: dynamics {gaps["dynamics"]:.3f} dex; '
            f'stellar sum {gaps["stellar_sum"]:.3f} dex (stellar change {gaps["stellar_sum"]-gaps["dynamics"]:+.3f} dex).')
        for f in ('dynamics','stellar_sum'):
            med=[lookup.loc[(p+'_'+f,kind)].log_median for p in ('flamingo','tng','combined')]
            position='between' if min(med[:2])<=med[2]<=max(med[:2]) else 'outside'
            lines.append(f'- Combined {f} median {med[2]:.3f} dex lies {position} the single-simulation medians '
                f'({med[0]:.3f}, {med[1]:.3f}); no posterior averaging is performed.')
        lines.append('')
    lines += ['## Effect of the approximate observational uncertainty','']
    for r in effects.itertuples(index=False):
        direction='broadens' if r.width68_change_dex>0 else 'narrows' if r.width68_change_dex<0 else 'leaves unchanged'
        lines.append(f'- {r.model}: median shift {r.median_shift_dex:+.3f} dex; '
            f'{direction} the 68% interval by {r.width68_change_dex:+.3f} dex ({r.width68_percentage_change:+.1f}%); '
            f'95% width change {r.width95_change_dex:+.3f} dex. '
            f'MC diagnostic: {"passed declared tolerance" if stability_pass[r.model] else "NOT ESTABLISHED; inspect stability table"}.')
    for f in ('dynamics','stellar_sum'):
        gap=abs(lookup.loc[('flamingo_'+f,'nominal')].log_median-lookup.loc[('tng_'+f,'nominal')].log_median)
        relevant=effects[effects.model.isin([p+'_'+f for p in ('flamingo','tng','combined')])]
        shift=relevant.median_shift_dex.abs().max(); width=relevant.width68_change_dex.abs().max()
        direction='larger' if gap>shift else 'no larger'
        lines.append(f'- {f}: nominal FLAMINGO–TNG median gap {gap:.3f} dex is {direction} than the largest '
            f'absolute observational median shift ({shift:.3f} dex). Largest absolute 68% width change is {width:.3f} dex; '
            'width change and median displacement are different summaries, not variance components.')
    lines += ['', '## Limitations', '',
        '- Missing joint observational covariance and shared IMF/SPS systematics; M31 asymmetric PDF is approximated.',
        '- Observational disk/bulge/aperture masses do not exactly equal all-bound simulation stellar masses.',
        '- Combined H0 is an explicit reference convention; no simulation-specific stellar shifts or Delta are used.',
        '- '+support_status+'. No observational realizations were removed. Marginal bounds do not establish joint support.',
        '- The input mixture is conditional on the stated approximate input distribution, not a demonstrated exact simulation-population Bayesian marginal.',
        '- Differences are descriptive; no preferred/correct simulation, consensus mass, or significance claim is selected.']
    return '\n'.join(lines)

if RUN_INFERENCE:
    findings=result_markdown(summary,effects,contrasts,stability_pass,SUPPORT_STATUS)
    (OUTPUT/'local_group_findings.md').write_text(findings,encoding='utf-8')
    # Recheck original scientific inputs after inference; no final-stage artifacts were changed.
    for relative,digest in INPUT_HASHES.items():
        assert sha256(PROJECT/relative)==digest, 'Input changed during application: '+relative
    for key in MODEL_KEYS:
        assert sha256(PROJECT/REGISTRY[key]/'model.pkl')==MANIFESTS[key]['model_sha256']
        assert sha256(PROJECT/REGISTRY[key]/'run_manifest.json')==EVIDENCE['run_manifest_sha256'][key]
    MANIFEST.update(status='complete',completed_at_utc=datetime.now(timezone.utc).isoformat(),
        mc_stability_pass=stability_pass,figure_names=FIGURE_NAMES,
        scientific_status='approximate scenario; '+('declared MC tolerance passed' if all(stability_pass.values()) else 'MC stability NOT established for all models'),
        output_sha256={p.name:sha256(p) for p in sorted(OUTPUT.iterdir()) if p.is_file() and p.name!='manifest.json'})
    write_json(OUTPUT/'manifest.json',MANIFEST)
    display(Markdown(findings))
    print('Completed output:',OUTPUT)
else:
    display(Markdown('**Local audit finished. No scientific posteriors have been generated.** Execute on Roihu using the companion script after reading the observation assumptions above.'))

## Optional execution

Use the reproduction notebook and the explicit `LG_RUN_RESEARCH_WORKFLOW=1` opt-in. For a new Local Group inference, execute this notebook with Papermill in the pinned research environment. The saved result notebooks do not require a new cluster run. Retain the fixed input archive and write the new executed notebook under `research/out/`.
